# graphArt_LM: the closing experiments on Hugging Face (RQ4-RQ6)

About 2.1 hours, capped at 2.5.

1. Scores the Phase 2 adapter (Qwen2.5-Coder-1.5B + LoRA) on real torchvision code (RQ4)
   and on the controlled-edit set (RQ5).
2. Trains a second adapter, for one epoch, with every program containing a dead branch
   held out, and scores it on the RQ6 probe set (RQ6); then the Phase 2 adapter on the
   same probe, if time is left.

**Before running**: *GPU T4 x2*, *Internet on*. Optionally upload the folder
**Upload the folder `qwen2.5-coder-1.5b-graphart-lora`** (from the Phase 2 zip) as a dataset
under *Add Input -> Upload*. Without it, step 1 is skipped. Then *Save Version → Save & Run All (Commit)*.

**When it finishes**, download `graphart_closing_hf.zip` from the *Output* tab.

Generated by `scripts/make_kaggle_notebook.py --closing-hf`. Edit that script, not this file.

In [ ]:
# FULL_ADAPTER     the Phase 2 adapter. Upload its folder as a Kaggle dataset (Add Input ->
#                  Upload). Without it, its evaluations are skipped -- unless RETRAIN_FULL,
#                  which adds ~1.6 h.
# HELDOUT_ADAPTER  a second adapter trained with every HELDOUT_CONSTRUCT program left out
#                  of train and val (RQ6), for HELDOUT_EPOCHS (one epoch: val loss had
#                  already reached 0.0003 after one in Phase 2).
# *_EVALS          (data set, reply cap) pairs, run in order: torchvision (RQ4), edits (RQ5),
#                  rq6 (the probe set). FULL_EVALS_IF_TIME runs last, only if time is left.
# TIME_BUDGET      the whole session, 2.5 h; work that would not fit is skipped, not cut short.
# VALIDATE_ONLY    True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; FULL_ADAPTER = "qwen2.5-coder-1.5b-graphart-lora"; HELDOUT_ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-no-dead-branch"; HELDOUT_CONSTRUCT = "dead_branch"; FULL_EVALS = [("torchvision", 3200), ("edits", None)]; HELDOUT_EVALS = [("rq6", None)]; FULL_EVALS_IF_TIME = [("rq6", None)]; RETRAIN_FULL = False; EPOCHS = 2.0; HELDOUT_EPOCHS = 1.0; HELDOUT_TRAIN_ARGS = ["--eval-steps", "35"]; SMOKE_STEPS = 8; TIME_BUDGET_HOURS = 2.5; TRAIN_ARGS = []; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "bc0822c0538bcb179f29764e64007db1af1746a1c6a15665d7991e6e8ebb805d"
BUNDLE = (
    "..."
    "v8vfj/73/wNfxsaOAJgwAA=="
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. The Phase 2 adapter: uploaded, or trained here

In [ ]:
import zipfile

GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
INPUT = Path(os.environ.get("GRAPHART_INPUT", "/kaggle/input"))

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package():
    # Results and both adapters, without optimizer-state checkpoints.
    with zipfile.ZipFile(WORK / "graphart_closing_hf.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", REPO / "lora" / FULL_ADAPTER, REPO / "lora" / HELDOUT_ADAPTER):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file() and "checkpoints" not in p.parts:
                    z.write(p, p.relative_to(REPO))

def train(*extra):
    return subprocess.run([sys.executable, "scripts/train_lora.py", "--base", BASE_MODEL,
                           *TRAIN_ARGS, *extra], env=GPU_ENV).returncode

SMOKED = False
def smoke():
    # Before the first training only: data check, then a short run that must learn.
    global SMOKED, TRAIN_ARGS
    if SMOKED:
        return
    if train("--out", "lora/smoke", "--check-only") != 0:
        raise SystemExit("the data check failed -- see the traceback above")
    code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS))
    if code in (3, 4):                      # not learning / NaN: maybe fp16, retry in fp32
        print(f"smoke run failed in fp16 (exit {code}) -- retrying in fp32", flush=True)
        code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS), "--fp32")
        if code == 0:
            TRAIN_ARGS = [*TRAIN_ARGS, "--fp32"]
    if code != 0:
        raise SystemExit(f"the smoke run failed (exit {code}) -- see the log above")
    SMOKED = True

def uploaded_adapter():
    for cfg in INPUT.rglob("adapter_config.json") if INPUT.exists() else []:
        if json.loads(cfg.read_text()).get("base_model_name_or_path") == BASE_MODEL \
                and (cfg.parent / "adapter_model.safetensors").exists():
            return cfg.parent
    return None

full = REPO / "lora" / FULL_ADAPTER
source = uploaded_adapter()
if source:
    shutil.copytree(source, full, dirs_exist_ok=True)
    print(f"using the uploaded adapter from {source}", flush=True)
elif RETRAIN_FULL:
    print("no adapter uploaded under /kaggle/input -- training it here (~1.6 h)", flush=True)
    smoke()
    if train("--out", f"lora/{FULL_ADAPTER}", "--epochs", str(EPOCHS),
             "--time-budget-hours", f"{max(remaining() / 3600 - 1, 0.5):.2f}") != 0:
        raise SystemExit("training the Phase 2 adapter failed -- see the log above")
else:
    print("NOTE: no Phase 2 adapter uploaded under /kaggle/input, and RETRAIN_FULL is False -- "
          "skipping its evaluations; RQ6 still runs", flush=True)
HAVE_FULL = (full / "adapter_config.json").exists()
package()

## 5. Score the Phase 2 adapter (RQ4, RQ5, all 160 test programs)

In [ ]:
def evaluate(adapter_dir, evals):
    for split, cap in evals:
        if remaining() < 900:
            print(f"time budget spent -- skipping {adapter_dir.name} on {split}")
            continue
        cmd = [sys.executable, "-m", "pipeline.run_prompting", "--backend", "hf", "--model", str(adapter_dir),
               "--prompt", "zero_shot", "--split", split, *(["--max-tokens", str(cap)] if cap else []), *EVAL_ARGS]
        print(f"\n##### {adapter_dir.name} on {split}   ({remaining() / 3600:.1f} h left)", flush=True)
        try:
            subprocess.run(cmd, env=GPU_ENV, timeout=remaining())
        except subprocess.TimeoutExpired:
            print("stopped by the time budget")
        package()

if HAVE_FULL:
    evaluate(full, FULL_EVALS)

## 6. Train the held-out adapter (RQ6)

In [ ]:
heldout = REPO / "lora" / HELDOUT_ADAPTER
smoke()
# Keep ~25 minutes for scoring the probe set afterwards.
if train("--out", f"lora/{HELDOUT_ADAPTER}", "--epochs", str(HELDOUT_EPOCHS),
         "--exclude-construct", HELDOUT_CONSTRUCT, *HELDOUT_TRAIN_ARGS,
         "--time-budget-hours", f"{max(remaining() / 3600 - 0.42, 0.25):.2f}") != 0:
    raise SystemExit("training the held-out adapter failed -- the Phase 2 results above are saved")
print((heldout / "train_summary.json").read_text())
package()

## 7. Score the held-out adapter, and list everything

In [ ]:
evaluate(heldout, HELDOUT_EVALS)
if HAVE_FULL:
    evaluate(full, FULL_EVALS_IF_TIME)        # the same probe, for a side-by-side
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:75s} n={m['n']} failed={m['failed']}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download graphart_closing_hf.zip")